# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name    : {metadata.name}")
print(f"Dataset Version : {metadata.version}")
print(f"Identifier      : {metadata.identifier}")
print(f"License         : {metadata.license}")
print(f"Description     : {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All identifiers are referenced by their `@id` fields.


In [ ]:
# List available RecordSets and their IDs
print('Available record sets:')
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id} (name: {rs.name})")

# We select the main tabular record set for review
record_set_id = metadata.record_sets[0].id
print(f"\nInspecting fields for RecordSet @id: {record_set_id}")

fields = metadata.record_sets[0].fields
for field in fields:
    print(f"  - Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# For this dataset, we extract from the main RecordSet
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for rsid in record_sets_ids:
    # Load all records (as dictionaries)
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

print(f"Loaded DataFrame columns for RecordSet @id: {record_sets_ids[0]}")
print(dataframes[record_sets_ids[0]].columns.tolist())
dataframes[record_sets_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All columns and entities referenced by their `@id`.

Let's:
- Select and filter the `@id` of the numeric field: age at diagnosis (`age_at_diagnosis`) if present.
- Normalize this numeric field.
- Group by a categorical field: e.g., sex (`sex`) if present.


In [ ]:
# --- EDA Parameters (set using field @id) ---
numeric_field_id = None
group_field_id = None
df = dataframes[record_sets_ids[0]]

# Let's discover field @id for common variables:
for field in metadata.record_sets[0].fields:
    if 'age' in field.name.lower():
        numeric_field_id = field.id
    if 'sex' in field.name.lower() or 'gender' in field.name.lower():
        group_field_id = field.id

if numeric_field_id is None:
    print("No age-related numeric field found. Using first numeric field.")
    for field in metadata.record_sets[0].fields:
        if field.data_type in ['Integer', 'Float', 'Number']:
            numeric_field_id = field.id
            break
if numeric_field_id is None:
    raise ValueError('No numeric field found in the dataset.')

print(f"Selected numeric field @id: {numeric_field_id}")
if group_field_id:
    print(f"Selected group field @id: {group_field_id}")
# Convert numeric column to numeric type if necessary
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: keep only records with age > 40 (typical for CRC), or pick threshold = 10
threshold = 40
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouped mean by the group field
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot a histogram for the age-at-diagnosis field, and if grouping is available, boxplots by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True, color='skyblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.grid(True)
plt.show()

# Boxplot by group if group_field_id exists
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, inspect, and analyze a FAIR² dataset defined by a Croissant schema. 

- We accessed all RecordSets, fields, and columns using their unique `@id`s, as recommended for consistent processing.
- We performed a basic exploratory data analysis on clinical data, showing filtering, normalization, grouping, and visualizations.
- This workflow enables seamless, reproducible ingestion and preparation of FAIR datasets in biomedical and clinical research contexts.
